# doc-extraction — OmniDocBench full benchmark (Kaggle, T4 GPU)

Thin orchestrator notebook — 10 steps, no extraction or evaluation logic
implemented here. Everything runs through the repo's own scripts
(`experiments/005_omnidocbench/run.py`,
`src/doc_extraction/evaluation/omnidocbench.py`). See
[`experiments/005_omnidocbench/README.md`](../README.md) for the full design,
the pinned OmniDocBench commit, and what has/hasn't been run locally, and
[`docs/kaggle.md`](../../../docs/kaggle.md) for the step-by-step setup this
notebook assumes.

**No private data.** Only the public OmniDocBench dataset is used here. This
repo's own `data/` (local/private sample documents) is gitignored and is not
part of this clone — never attach it as a Kaggle input.

**CPU pipeline vs. GPU backend vs. benchmark evaluator** — three different
things that are easy to conflate:
- The **pipeline** (`baseline`, `docling` backends) runs on **CPU** by
  default (`configs/cpu.yaml`, `device: cpu`) whether or not a GPU is
  attached — neither backend currently requests CUDA.
- The **T4 GPU** this notebook checks for (Step 2) is only actually used
  once a GPU-requesting backend exists (see `docs/backends.md`); today it
  buys faster page rendering/model inference *if* a backend uses it, not
  automatically.
- The **OmniDocBench evaluator** (Step 4/8) is a separate, CPU-only, Python
  \<3.12 subprocess that scores predictions — it never touches the GPU.

## Step 1 — Check Python

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "doc_extraction needs Python 3.10+"

## Step 2 — Check GPU

Diagnostic only (see the CPU/GPU/evaluator note above) — nothing later in
this notebook currently changes behavior based on this.

In [ ]:
import torch

has_cuda = torch.cuda.is_available()
print(f"CUDA available: {has_cuda}")
if has_cuda:
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU visible — check Settings > Accelerator is set to GPU T4 x2 (or T4 x1).")

## Step 3 — Clone repo

In [ ]:
%cd /kaggle/working
!git clone https://github.com/<YOUR_USERNAME>/doc-extraction.git
%cd doc-extraction

## Step 4 — Install deps

Two independent installs: this project's own package, and the OmniDocBench
evaluator (a separate project, pinned to the exact commit this repo's
adapter was verified against — see the parent README's "Upstream, pinned
exactly" table).

In [ ]:
# This project. Kaggle's default Python satisfies the package's own
# constraints directly.
!pip install -e ".[docling,tables]" -q

# OmniDocBench evaluator: cloned outside the repo (its own Apache-2.0
# project, not vendored) and pinned to the commit the adapter targets.
!git clone https://github.com/opendatalab/OmniDocBench.git .external/OmniDocBench
%cd .external/OmniDocBench
!git checkout 193627ae9e97d89188468ed1ee3b7a856ff76044
%cd /kaggle/working/doc-extraction

# The evaluator requires Python >=3.10,<3.12. If the attached Kaggle image's
# default Python is 3.12+, create an isolated env instead and pass
# --omnidoc-python .venv-omnidoc/bin/python to run.py in Steps 7-8:
#   !python -m venv .venv-omnidoc && .venv-omnidoc/bin/pip install -e .external/OmniDocBench
!pip install -e .external/OmniDocBench -q

## Step 5 — Locate OmniDocBench dataset

Option A (recommended): attach the dataset as a Kaggle Dataset input —
it appears under `/kaggle/input/<dataset-name>/`. Option B: download the
official dataset directly from its public source. Either way, only public
OmniDocBench data — never this repo's own `data/`.

In [ ]:
import os

# Option A: attached Kaggle Dataset — edit <dataset-name> to match what you
# attached via Add Input in the notebook editor.
DATASET_PATH = "/kaggle/input/<dataset-name>"

# Option B (uncomment if not attaching a Dataset): download the small
# official demo set bundled with the evaluator clone instead.
# DATASET_PATH = "/kaggle/working/doc-extraction/.external/OmniDocBench/demo_data/omnidocbench_demo"

OUTPUT_ROOT = "/kaggle/working/results"

# Keep model downloads on the working volume, not the small root volume —
# same reasoning as the dev machine's cache redirection (docs/setup.md).
os.environ.setdefault("HF_HOME", "/kaggle/working/.cache/huggingface")
os.environ.setdefault("DOCLING_ARTIFACTS_PATH", "/kaggle/working/.cache/docling")
os.environ.setdefault("XDG_CACHE_HOME", "/kaggle/working/.cache")

## Step 6 — Print dataset path

In [ ]:
assert os.path.exists(DATASET_PATH), f"DATASET_PATH does not exist: {DATASET_PATH} — check the attached input's mount path"
print(f"DATASET_PATH = {DATASET_PATH}")
print(sorted(os.listdir(DATASET_PATH))[:20])

## Step 7 — Run small smoke test

Validate the integration on a small, deterministic subset before spending
GPU/CPU time on the full run — same principle as the local CPU validation
(see the main README's "Local validation").

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --subset 20 \
    --match-workers 2

## Step 8 — Run full experiment

Omit `--subset` for the full dataset. `--match-workers`: keep to roughly
1/3-1/2 of the instance's CPU count (upstream's own guidance, to avoid
deadlocks/OOM in its worker pools) — the evaluator itself is CPU-bound
regardless of the attached accelerator.

`docling` is included as a second arm for comparison; both currently run
on CPU (see the CPU/GPU/evaluator note at the top). Do not claim a backend
used the T4 unless Step 2 showed CUDA available *and* the backend actually
requests `device: cuda` (see `configs/gpu.yaml` — documented, unvalidated).

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline \
    --match-workers 4

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend docling \
    --output {OUTPUT_ROOT}/docling \
    --match-workers 4

## Step 9 — Print metrics

In [ ]:
from pathlib import Path

for backend_dir in sorted(Path(OUTPUT_ROOT).glob("*")):
    report = backend_dir / "report.md"
    if report.exists():
        print(f"===== {backend_dir.name} =====")
        print(report.read_text(encoding="utf-8"))
        print()

## Step 10 — Save results

`/kaggle/working` persists as the notebook's output and can be downloaded
from the output panel after the session ends. To fold results back into
the repo's own history, copy just the small committed-shape files
(`report.md`, `metrics.json`, `runtime.json`, `run_metadata.json` — not
`predictions/` or the evaluator's raw debug dumps) into
`experiments/005_omnidocbench/results/<backend>/`, matching the local-run
convention described in the parent README's "Files" section.

In [ ]:
import shutil

for backend in ("baseline", "docling"):
    src = Path(OUTPUT_ROOT) / backend
    if not src.exists():
        continue
    for name in ("report.md", "metrics.json", "runtime.json", "run_metadata.json"):
        f = src / name
        if f.exists():
            print(f"kept for output panel: {f}")
# Files above remain under /kaggle/working; download them from the output
# panel, then copy locally into experiments/005_omnidocbench/results/.